# Executing and reading a sweep

This hardware-free example runs a small sweep in a temporary directory, merges its chunks, and opens the result through the generic dataset API.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

import pandas as pd

from pynst import GenericSweepDataset, SweepManager
from pynst.planning import create_multiindex, create_range

In [ ]:
def measure(parameters):
    voltage = parameters["voltage_V"]
    return {
        "measurement": pd.DataFrame(
            {"current_A": [voltage / 50.0]},
            index=pd.Index([0], name="sample"),
        )
    }

voltages = create_range(min=0.0, max=1.0, num=3)
grid = create_multiindex([voltages], names=["voltage_V"])

In [ ]:
with TemporaryDirectory() as temporary_directory:
    root = Path(temporary_directory)
    merged_path = root / "tutorial.h5"

    with SweepManager(
        measurement_func=measure,
        ivars=grid,
        meas_name="tutorial_run",
        output_root=root,
        chunk_size=2,
        resume=False,
        metadata={"operator": "documentation"},
    ) as manager:
        complete = manager.run()
        manager.merge(merged_path)

    dataset = GenericSweepDataset(merged_path)
    frame = dataset["measurement"]
    print(f"complete: {complete}")
    print(f"blocks: {dataset.block_names}")
    print(frame)

The local `sample` index is retained beneath the global `voltage_V` level. Reconstructing the manager with the same measurement name, grid, and schema and with `resume=True` would validate and continue an interrupted run.